# Monte Carlo Value-at-Risk (VaR)

This notebook demonstrates how to estimate VaR using a Monte Carlo simulation.

Here portfolio returns are simulated assuming a multivariate normal distribution, then the Monte Carlois VaR compared with Historical and Parametric VaR.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

from src.var_methods import historical_var, parametric_var, monte_carlo_var

## Load Data

In [ ]:
tickers = ["SPY", "BND", "GLD"]
data = yf.download(tickers, start="2020-01-01", end="2025-01-01")["Close"]
log_returns = np.log(data / data.shift(1)).dropna()

weights = np.array([0.5, 0.3, 0.2])
portfolio_value = 1_000_000

## Monte Carlo Simulation VaR

In [ ]:
alpha = 0.05

hist_var = historical_var(log_returns, weights, alpha, portfolio_value)
param_var = parametric_var(log_returns, weights, alpha, portfolio_value)
mc_var = monte_carlo_var(log_returns, weights, alpha, sims=20000, portfolio_value=portfolio_value, seed=42)

print(f"Historical 95% VaR: ${hist_var:,.2f}")
print(f"Parametric 95% VaR: ${param_var:,.2f}")
print(f"Monte Carlo 95% VaR: ${mc_var:,.2f}")

## Visualization

In [ ]:
portfolio_returns = log_returns.dot(weights)

plt.figure(figsize=(10,6))
plt.hist(portfolio_returns, bins=50, alpha=0.7, label="Empirical returns")
plt.axvline(-hist_var/portfolio_value, color="blue", linestyle="--", label="Historical VaR")
plt.axvline(-param_var/portfolio_value, color="green", linestyle="--", label="Parametric VaR")
plt.axvline(-mc_var/portfolio_value, color="red", linestyle="--", label="Monte Carlo VaR")
plt.legend()
plt.title("Comparison of VaR Methods (95%)")
plt.show()

# Save figure for README
plt.savefig("docs/figures/var_comparison.png", dpi=150, bbox_inches="tight")